In [2]:
!git clone https://github.com/ashukla21/Fast-dLLM.git
%cd Fast-dLLM
!git fetch origin feat/layer-skip-dream
!git checkout feat/layer-skip-dream

Cloning into 'Fast-dLLM'...
remote: Enumerating objects: 445, done.
remote: Counting objects: 100% (232/232), done.
remote: Compressing objects: 100% (143/143), done.
remote: Total 445 (delta 137), reused 162 (delta 88), pack-reused 213 (from 2)
Receiving objects: 100% (445/445), 122.40 MiB | 14.85 MiB/s, done.
Resolving deltas: 100% (221/221), done.
/content/Fast-dLLM
From https://github.com/ashukla21/Fast-dLLM
 * branch            feat/layer-skip-dream -> FETCH_HEAD
Branch 'feat/layer-skip-dream' set up to track remote branch 'feat/layer-skip-dream' from 'origin'.
Switched to a new branch 'feat/layer-skip-dream'


In [3]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/Fast-dLLM-logs

Mounted at /content/drive


In [4]:
# Fresh install that matches CUDA 12.1 in Colab
!pip -q install -U "numpy<2"  # keeps older C-extensions happy
!pip -q install --upgrade --no-cache-dir \
  torch==2.4.0+cu121 torchvision==0.19.0+cu121 torchaudio==2.4.0+cu121 \
  --index-url https://download.pytorch.org/whl/cu121

# Core libs used in the repo + your scripts
!pip -q install "transformers==4.43.3" "accelerate==0.33.0" einops sentencepiece tiktoken gradio safetensors

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 95.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 207.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 254.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━

In [5]:
import torch, numpy
print(torch.__version__)
print(numpy.__version__)
print("CUDA:", torch.version.cuda)
x = torch.ones(1, device="cuda"); print("OK:", x.device)

2.8.0+cu126
2.0.2
CUDA: 12.6
OK: cuda:0


In [6]:
import sys, os
if '.' not in sys.path:
    sys.path.insert(0, '.')
os.environ['PYTHONPATH'] = '.'

In [ ]:
!PYTHONPATH=. python scripts/run_similarity_llada.py \
  --out-dir /content/drive/MyDrive/Fast-dLLM-logs/llada_baseline \
  --steps 16 \
  --gen-length 64 \
  --block-size 32 \
  --prompt "Question: A charcoal grill burns fifteen coals to ash every twenty minutes of grilling. The grill ran for long enough to burn three bags of coals. Each bag of coal contains 60 coals. How long did the grill run?"


tokenizer_config.json: 51.7kB [00:00, 118MB/s]
tokenizer.json: 6.10MB [00:00, 152MB/s]
special_tokens_map.json: 100% 747/747 [00:00<00:00, 5.74MB/s]
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
config.json: 1.39kB [00:00, 8.98MB/s]
model.safetensors.index.json: 24.9kB [00:00, 104MB/s]
model-00001-of-00006.safetensors:   0% 0.00/2.11G [00:00<?, ?B/s]
model-00001-of-00006.safetensors:   0% 835k/2.11G [00:02<1:36:58, 362kB/s]
model-00001-of-00006.safetensors:   6% 135M/2.11G [00:02<00:26, 73.2MB/s] 
model-00001-of-00006.safetensors:  19% 403M/2.11G [00:02<00:06, 256MB/s] 
model-00001-of-00006.safetensors:  29% 604M/2.11G [00:02<00:03, 399MB/s]
model-00001-of-00006.safetensors:  35% 739M/2.11G [00:02<00:02, 475MB/s]
model-00001-of-00006.safetensors:  41% 873M/2.11G [00:06<00:11, 110MB/s]
model-00001-of-00006.safetensors:  48% 1.01G/2.11G [00:06<00:07, 149MB/s]
model-00001-of-00006.safetensors:  54% 1.14G/2.11G [00:06<00:04, 200MB/s

In [6]:
!pip install -q huggingface_hub
!huggingface-cli login

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) y
Token is valid (permission: read).
The token `colab-access` has been saved to /root/.cache/huggingface/stored_tokens
Cannot authenticate through git-credent

In [7]:
from huggingface_hub import snapshot_download
import os

# path in Drive to store the model
target = "/content/drive/MyDrive/checkpoints/Dream-v0-Base-7B"
os.makedirs(target, exist_ok=True)

path = snapshot_download(
    repo_id="Dream-org/Dream-v0-Base-7B",
    local_dir=target,
    local_dir_use_symlinks=False
)

print("✅ Downloaded Dream checkpoint to:", path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:982: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

✅ Downloaded Dream checkpoint to: /content/drive/MyDrive/checkpoints/Dream-v0-Base-7B


In [ ]:
!pip install "transformers==4.43.3"

  Using cached transformers-4.43.3-py3-none-any.whl.metadata (43 kB)
Using cached transformers-4.43.3-py3-none-any.whl (9.4 MB)
  Attempting uninstall: transformers
    Found existing installation: transformers 4.42.4
    Uninstalling transformers-4.42.4:
      Successfully uninstalled transformers-4.42.4


In [ ]:
!PYTHONPATH=. python scripts/run_similarity_dream.py \
  --out-dir /content/drive/MyDrive/Fast-dLLM-logs/dream_baseline \
  --steps 16 \
  --block-size 32 \
  --local-model-dir /content/drive/MyDrive/checkpoints/Dream-v0-Base-7B \
  --prompt "Question: A charcoal grill burns fifteen coals to ash every twenty minutes of grilling. The grill ran for long enough to burn three bags of coals. Each bag of coal contains 60 coals. How long did the grill run?"

[Dream] Loading from /content/drive/MyDrive/checkpoints/Dream-v0-Base-7B
Loading checkpoint shards: 100% 4/4 [00:59<00:00, 14.83s/it]
[Dream] tapped 28 layers
[Dream] step 00: saved within_step_cosine_step0.pt (shape=(28, 28))
[Dream] step 01: saved within_step_cosine_step1.pt (shape=(28, 28))
[Dream] step 02: saved within_step_cosine_step2.pt (shape=(28, 28))
[Dream] step 03: saved within_step_cosine_step3.pt (shape=(28, 28))
[Dream] step 04: saved within_step_cosine_step4.pt (shape=(28, 28))
[Dream] step 05: saved within_step_cosine_step5.pt (shape=(28, 28))
[Dream] step 06: saved within_step_cosine_step6.pt (shape=(28, 28))
[Dream] step 07: saved within_step_cosine_step7.pt (shape=(28, 28))
[Dream] step 08: saved within_step_cosine_step8.pt (shape=(28, 28))
[Dream] step 09: saved within_step_cosine_step9.pt (shape=(28, 28))
[Dream] step 10: saved within_step_cosine_step10.pt (shape=(28, 28))
[Dream] step 11: saved within_step_cosine_step11.pt (shape=(28, 28))
[Dream] step 12: saved 

In [ ]:
import torch, glob, os
d = "/content/drive/MyDrive/Fast-dLLM-logs/dream_baseline"
pts = sorted(glob.glob(os.path.join(d, "within_step_cosine_step*.pt")))
print("count:", len(pts), "first:", os.path.basename(pts[0]), "last:", os.path.basename(pts[-1]))

# Inspect a couple of steps
for p in [pts[0], pts[len(pts)//2], pts[-1]]:
    M = torch.load(p, map_location="cpu")
    print(os.path.basename(p), "shape:", tuple(M.shape),
          "min/max:", float(M.min()), float(M.max()),
          "diag_mean:", float(M.diag().mean()))
    # symmetry + identity-ish diagonal checks
    sym_err = (M - M.T).abs().max().item()
    diag_err = (M.diag() - torch.ones(M.size(0))).abs().max().item()
    print("  symmetry_err:", sym_err, "diag_≈1_err:", diag_err)


count: 16 first: within_step_cosine_step0.pt last: within_step_cosine_step9.pt
within_step_cosine_step0.pt shape: (28, 28) min/max: 0.02950548566877842 1.0000014305114746 diag_mean: 1.0000009536743164
  symmetry_err: 0.0 diag_≈1_err: 1.430511474609375e-06
within_step_cosine_step2.pt shape: (28, 28) min/max: 0.02950548566877842 1.0000014305114746 diag_mean: 1.0000009536743164
  symmetry_err: 0.0 diag_≈1_err: 1.430511474609375e-06
within_step_cosine_step9.pt shape: (28, 28) min/max: 0.052070725709199905 1.0000014305114746 diag_mean: 1.000001072883606
  symmetry_err: 0.0 diag_≈1_err: 1.430511474609375e-06


/tmp/ipython-input-2697590240.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  M = torch.load(p, map_location="cpu")


In [15]:
import sys, os
sys.path.append('/content/Fast-dLLM/dream')
sys.path.append('/content/Fast-dLLM/scheduling')
sys.path.append('/content/Fast-dLLM/utils')
sys.path.append('/content/Fast-dLLM/llada')
!pip install -e .
sys.path.append('/content/Fast-dLLM')
os.listdir()

Obtaining file:///content/Fast-dLLM
ERROR: file:///content/Fast-dLLM does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


['asset',
 'llada',
 'scripts.zip',
 'Fast-dLLM-logs',
 'scheduling.zip',
 'dream.zip',
 'scheduling',
 '.git',
 'README.md',
 'requirements.txt',
 'dream',
 'llada.zip',
 'scripts',
 'Fast-dLLM-logs.zip',
 'v2',
 'utils',
 'CONTRIBUTING.md',
 'utils.zip',
 '.gitignore',
 'LICENSE']

In [11]:
!python scripts/run_dream.py \
  --local-model-dir /content/drive/MyDrive/checkpoints/Dream-v0-Base-7B \
  --steps 128 --max-new-tokens 64 \
  --mask-token-id 151666 \
  --temperature 0.0 \
  --prompt "Question: A charcoal grill burns fifteen coals to ash every twenty minutes of grilling. The grill ran for long enough to burn three bags of coals. Each bag of coal contains 60 coals. How long did the grill run?" \
  --schedule-path /content/drive/MyDrive/Fast-dLLM-logs/dream_baseline/skip_schedule_tau0.999.json \
  --output-dir /content/drive/MyDrive/Fast-dLLM-logs/runs/$(date +%Y%m%d_%H%M%S) \
  --save-history

The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
Loading checkpoint shards: 100% 4/4 [03:02<00:00, 45.55s/it]
used steps: 128
used time: 6.777429103851318
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Question: A charcoal grill burns fifteen coals to ash every twenty minutes of grilling. The grill ran for long enough to burn three bags of coals. Each bag of coal contains 60 coals. How long did the grill run?<|im_end|>
<|im_start|>assistant
11 co co coals1 co co co co co co co co co co co co co co 1 co co0 co co co co co co bag co 0als, 000000000000 00000000 000


In [10]:
!PYTHONPATH=. python scripts/run_llada.py \
  --steps 128 \
  --max-new-tokens 128 \
  --mask-token-id 151666 \
  --temperature 0.0 \
  --dtype fp16 \
  --prompt "Question: A charcoal grill burns fifteen coals to ash every twenty minutes of grilling. The grill ran for long enough to burn three bags of coals. Each bag of coal contains 60 coals. How long did the grill run?" \
  --schedule-path /content/drive/MyDrive/Fast-dLLM-logs/llada_baseline/skip_schedule_tau0.9995.json \
  --tokenizer-id /content/drive/MyDrive/checkpoints/Dream-v0-Base-7B \
  --output-dir /content/drive/MyDrive/Fast-dLLM-logs/runs_llada/$(date +%Y%m%d_%H%M%S) \
  --save-history

Traceback (most recent call last):
  File "/content/Fast-dLLM/scripts/run_llada.py", line 163, in <module>
    main()
  File "/content/Fast-dLLM/scripts/run_llada.py", line 118, in main
    out = model.diffusion_generate(
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/Fast-dLLM/llada/model/modeling_llada.py", line 1699, in diffusion_generate
    outputs = self.forward(
              ^^^^^^^^^^^^^
  File "/content/Fast-dLLM/llada/model/modeling_llada.py", line 1594, in forward
    outputs = self.model.forward(
              ^^^^^^^^^^^^^^^^^^^
  File "/content/Fast-dLLM/llada/model/modeling_llada.py", line 1491, in forward
    x, cache = block(
               ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1553, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1562, in _call_impl
    return forward_c

In [61]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()


In [8]:
%env PYTORCH_CUDA_ALLOC_CONF=max_split_size_mb:64,garbage_collection_threshold:0.8


env: PYTORCH_CUDA_ALLOC_CONF=max_split_size_mb:64,garbage_collection_threshold:0.8


In [9]:
%env PYTORCH_CUDA_ALLOC_CONF=

env: PYTORCH_CUDA_ALLOC_CONF=


In [1]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:128
import torch
torch.set_float32_matmul_precision("high")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)


env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:128


In [65]:
!nvidia-smi
!kill -9 675106 694656
!nvidia-smi


Wed Oct 15 04:03:46 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             50W /  400W |     501MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [62]:
!nvidia-smi


Wed Oct 15 03:56:09 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             50W /  400W |     501MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [59]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True


env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True


In [ ]:
import json, time
from datetime import datetime

# Decode the final output text
decoded = tok.decode(seq[0], skip_special_tokens=True)

# Create metadata
result = {
    "timestamp": datetime.now().isoformat(),
    "model": args.model if hasattr(args, "model") else getattr(args, "local_model_dir", None),
    "prompt": args.prompt,
    "decoded_output": decoded,
    "schedule_path": args.schedule_path,
    "steps": args.steps,
    "max_new_tokens": args.max_new_tokens,
    "mask_token_id": args.mask_token_id,
    "temperature": args.temperature,
    "top_p": args.top_p,
    "top_k": args.top_k,
    "output_dir": str(Path(args.output_dir).resolve()) if hasattr(args, "output_dir") else None,
    "used_time_seconds": time.time(),  # replace this if you track start/end time
}

# Ensure output directory exists
outdir = Path(args.output_dir)
outdir.mkdir(parents=True, exist_ok=True)

# Save as JSON
out_path = outdir / "run_summary.json"
with open(out_path, "w") as f:
    json.dump(result, f, indent=2)

print(f"\n✅ Run summary saved to: {out_path}")
print(f"📝 Decoded output:\n{decoded}")

In [27]:
from transformers import AutoTokenizer

# Load your Dream tokenizer (same path or HF model as your local model)
tok = AutoTokenizer.from_pretrained("/content/drive/MyDrive/checkpoints/Dream-v0-Base-7B", trust_remote_code=True)

print("✅ Tokenizer loaded successfully.\n")

# 1️⃣ Show its mask token info
print(f"mask_token_id: {tok.mask_token_id}")
print(f"mask_token: {tok.mask_token}")

# 2️⃣ Show its BOS/EOS tokens (used for correct prompt wrapping)
print(f"bos_token: {tok.bos_token} (id={tok.bos_token_id})")
print(f"eos_token: {tok.eos_token} (id={tok.eos_token_id})")

# 3️⃣ Check if the tokenizer supports the chat template API
if hasattr(tok, "apply_chat_template"):
    messages = [{"role": "user", "content": "How many minutes are in two hours?"}]
    prompt_txt = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    print("\n🗣️ Chat-formatted prompt example:\n")
    print(prompt_txt)
else:
    print("\n⚠️ This tokenizer does not support apply_chat_template(). You'll need to manually format the prompt using <|im_start|>user ... etc.")


✅ Tokenizer loaded successfully.

mask_token_id: 151666
mask_token: <|mask|>
bos_token: <|beginoftext|> (id=151665)
eos_token: <|endoftext|> (id=151643)

🗣️ Chat-formatted prompt example:

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
How many minutes are in two hours?<|im_end|>
<|im_start|>assistant

